# CARpsy — OBDient Fine-tuning en Google Colab

**Modelo base:** Qwen3-0.6B  
**Método:** LoRA con Unsloth (2-5x más rápido que HuggingFace estándar)  
**Dataset:** 12,247 ejemplos de diagnóstico OBD-II en formato ChatML  
**Salida:** Adapter GGUF listo para usar con llama.cpp / QVAC Fabric

> **Requisito:** Runtime → Change runtime type → GPU (T4)

In [ ]:
# CELDA 1 — Verificar GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU — ve a Runtime > Change runtime type > GPU')

In [ ]:
# CELDA 2 — Instalar Unsloth (~2-3 minutos)
!pip install unsloth --quiet
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo --quiet
print('✅ Unsloth instalado')

In [ ]:
# CELDA 3 — Montar Google Drive (guarda checkpoints y adapter)
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/CARpsy/qwen3-0.6b/adapter', exist_ok=True)
os.makedirs('/content/drive/MyDrive/CARpsy/qwen3-0.6b/checkpoints', exist_ok=True)
print('✅ Google Drive montado')
print('📁 Adapter:      /content/drive/MyDrive/CARpsy/qwen3-0.6b/adapter/')
print('📁 Checkpoints:  /content/drive/MyDrive/CARpsy/qwen3-0.6b/checkpoints/')

In [ ]:
# CELDA 4 — Clonar repo CARpsy desde GitHub
import os

REPO_URL = 'https://github.com/gazzimon/CARpsy.git'
REPO_DIR = '/content/CARpsy'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
    print('✅ Repo actualizado')
else:
    !git clone {REPO_URL} {REPO_DIR}
    print('✅ Repo clonado')

# Verificar dataset
train_path = f'{REPO_DIR}/data/splits/train.jsonl'
lines = open(train_path).readlines()
print(f'✅ Dataset listo: {len(lines):,} ejemplos de entrenamiento')

In [ ]:
# CELDA 5 — Cargar modelo Qwen3-0.6B con Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = 'unsloth/Qwen3-0.6B',
    max_seq_length = 512,
    dtype          = None,       # Auto: float16 en T4, bfloat16 en A100
    load_in_4bit   = True,       # Cabe en T4 (16 GB VRAM)
)
print(f'✅ Modelo cargado | VRAM usada: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

In [ ]:
# CELDA 6 — Configurar LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r               = 8,
    lora_alpha      = 16,
    lora_dropout    = 0.05,
    target_modules  = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias            = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state    = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA configurado | Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}%)')

In [ ]:
# CELDA 7 — Preparar dataset ChatML
import json
from datasets import Dataset

REPO_DIR   = '/content/CARpsy'
train_path = f'{REPO_DIR}/data/splits/train.jsonl'

raw       = [json.loads(l) for l in open(train_path) if l.strip()]
formatted = [{'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)} for ex in raw]
dataset   = Dataset.from_list(formatted)

print(f'✅ Dataset preparado: {len(dataset):,} ejemplos')
print('\nEjemplo:')
print(dataset[0]['text'][:300])

In [ ]:
# CELDA 8 — Entrenar
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import glob

CHECKPOINT_DIR = '/content/drive/MyDrive/CARpsy/qwen3-0.6b/checkpoints'

# Detectar checkpoint más reciente (sort numérico, no alfabético)
checkpoints = glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*')
checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f'▶ Resumiendo desde: {resume_from}')
else:
    print('▶ Entrenamiento desde cero')

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = 'text',
    max_seq_length     = 512,
    dataset_num_proc   = 2,
    args = SFTConfig(
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        num_train_epochs             = 3,
        learning_rate                = 2e-4,
        weight_decay                 = 1e-2,
        lr_scheduler_type            = 'cosine',
        warmup_steps                 = 50,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 50,
        save_steps                   = 100,
        save_total_limit             = 3,
        output_dir                   = CHECKPOINT_DIR,
        optim                        = 'adamw_8bit',
        seed                         = 42,
        report_to                    = 'none',
        max_seq_length               = 512,
    ),
)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# CELDA 9 — Guardar adapter en Google Drive
adapter_path = '/content/drive/MyDrive/CARpsy/qwen3-0.6b/adapter/carpsy-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Adapter guardado en: {adapter_path}')

In [ ]:
# CELDA 10 — Exportar a GGUF Q4_K_M (compatible con llama.cpp / QVAC Fabric)
import os

gguf_path = '/content/drive/MyDrive/CARpsy/adapter/carpsy-adapter'
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method='q4_k_m')

# Unsloth guarda los GGUFs en subcarpeta _gguf
gguf_dir = f'{gguf_path}_gguf'
if os.path.exists(gguf_dir):
    gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith('.gguf')]
else:
    gguf_files = [f for f in os.listdir(os.path.dirname(gguf_path)) if f.endswith('.gguf')]
    gguf_dir = os.path.dirname(gguf_path)

print('✅ Archivos GGUF generados:')
for f in gguf_files:
    size = os.path.getsize(f'{gguf_dir}/{f}') / 1024**2
    print(f'  {gguf_dir}/{f} — {size:.1f} MB')
print('\n🎉 Fine-tuning completo. Descarga el .gguf desde Google Drive.')

## Próximos pasos

1. Descargá `carpsy-adapter-unsloth.Q4_K_M.gguf` desde Google Drive
2. Colocalo en `CARpsy/output/adapter/`
3. Corré la evaluación:
   ```
   python scripts/05_evaluate_adapter.py
   ```